# K-Means paralelo: memoria compartida vs. memoria distribuida

**Evidencia de producto · Cómputo de Alto Rendimiento · Ingeniería en Datos e IA (7.º cuatrimestre)**
Autor: Jonathan Israel Gonzalez Ruiz · Proyecto elegido: **P1 – K-Means paralelo desde cero**

Este notebook es **solo explicativo**: documenta el diseño, la metodología y la lectura de los resultados.
**No ejecuta experimentos**; únicamente lee los archivos que ya generó `experimentos/ejecutar_experimentos.py`
en `resultados/`. El código vive en los scripts del repositorio.

Objetivo de la evidencia: diseñar, implementar y evaluar una solución paralela a un problema real de datos,
demostrando con mediciones cuánto mejora el rendimiento frente a la versión secuencial y explicando por qué.

## 1. El problema y qué se paraleliza

Algoritmo de Lloyd con `n` puntos en `R^d` y `k` clusters. Cada iteración tiene dos fases:

| Fase | Qué hace | Costo | ¿Paralelizable? |
|---|---|---|---|
| **Asignación + acumulación** | cada punto va al centroide más cercano y se suman sus coordenadas al acumulador de su cluster | `O(n·k·d)` | **Sí**, los puntos son independientes (paralelismo de datos) |
| **Actualización de centroides** | nuevo centroide = media del cluster; chequeo de desplazamiento | `O(k·d)` | No (es la parte serial) |

Como `n ≫ k` en este experimento, casi todo el tiempo está en la primera fase. La segunda fase, más la
reducción/comunicación entre trabajadores, forma la **fracción serial** `1 − f` que limita el speedup
según la Ley de Amdahl.

## 2. Las tres versiones

Las tres comparten **el mismo núcleo compilado** (`comun/kmeans_comun.py::bloque`), los mismos datos
sintéticos y la misma inicialización de centroides. Así, cualquier diferencia de tiempo se debe al modelo de
paralelismo y no a que una versión sea otro algoritmo.

| Versión | Carpeta | Modelo | Reparto del trabajo |
|---|---|---|---|
| Secuencial (`T1`) | `secuencial/` | 1 hilo | referencia de correctitud y de tiempo |
| Paralela A | `paralelo_memoria_compartida/` | **Memoria compartida** (hilos, numba `prange`) | bloques estáticos de filas; cada hilo con **acumuladores privados** (sin locks ni carreras) y reducción final |
| Paralela B | `paralelo_memoria_distribuida/` | **Memoria distribuida** (procesos MPI, `mpi4py`) | `Scatterv` de bloques de filas desde el rank 0; **una `Allreduce`** de `k·d + k` valores por iteración |

Detalles de diseño que importan en la defensa:

- **Acumuladores privados** en memoria compartida: evita sincronizar en el bucle caliente; el costo es una
  reducción de `hilos × (k·d + k)` valores, despreciable frente a `n·k·d`.
- **Una sola `Allreduce` por iteración** en MPI (sumas y cuentas empaquetadas en un único buffer). Todos los
  procesos calculan el mismo centroide nuevo, así que no hace falta un `Bcast` adicional.
- **Reparto equilibrado a ±1 fila** (`lo = r·n // p`), también cuando `n` no es divisible entre `p`.
- En MPI cada proceso usa **1 hilo de numba**, para que el paralelismo provenga solo de MPI.

## 3. Verificación de correctitud

`experimentos/verificar_correctitud.py` compara contra la secuencial: centroides (`rtol = 1e-9`), etiquetas
(**igualdad exacta**), inercia e iteraciones. Se prueba con 1, 2, 3, 4 y 8 hilos y con 1, 2, 3, 4 y 7 procesos
MPI (cantidades que **no** dividen a `n`, para ejercitar el reparto desigual), y además contra scikit-learn como
verificación independiente. Las diferencias esperables (~1e-14) vienen solo del **orden de las sumas** en coma
flotante, que cambia con el reparto.

In [ ]:
from pathlib import Path

RES = Path("resultados")
if not RES.exists():           # permite abrirlo desde la carpeta del repo o desde /experimentos
    RES = Path("..") / "resultados"
print(RES.joinpath("verificacion_correctitud.txt").read_text(encoding="utf-8"))

## 4. Metodología de medición

- **Carga fija**: 25 iteraciones exactas (`tol = 0`), para que todas las versiones hagan el mismo trabajo.
- **Qué se cronometra**: desde que los datos existen hasta tener los centroides finales. La compilación JIT se
  calienta antes y no cuenta. En MPI el tiempo **incluye el `Scatterv` inicial** (repartir datos es un costo real
  de memoria distribuida) y es el del proceso más lento (`Barrier` antes y después).
- **Repeticiones**: ≥ 3 por configuración (5 por defecto); se reporta media y desviación estándar.
- **Base de comparación**: `S(p) = T1 / Tp`, con `T1` el tiempo de la versión **secuencial** (no la paralela con p = 1).
- **Fórmulas**: `S(p) = T1/Tp`, `E(p) = S(p)/p`, y Amdahl `S(p) ≤ 1 / ((1−f) + f/p)`.
- Cada configuración corre en un **proceso aparte** para aislar memoria y variables de entorno.

## 5. Equipo de cómputo y resultados

Todo lo siguiente se **lee de `resultados/`**; si cambian los experimentos, el notebook se actualiza solo.

In [ ]:
import json

specs = json.loads(RES.joinpath("specs_equipo.json").read_text(encoding="utf-8"))
for clave in ("cpu_modelo", "nucleos_por_socket", "hilos_por_nucleo", "nucleos_logicos",
              "ram_gb", "cache_L3", "gpu", "mpi", "sistema"):
    if clave in specs:
        print(f"{clave:>20}: {specs[clave]}")
print(f"{'versiones':>20}: {specs.get('versiones')}")

In [ ]:
import csv
from IPython.display import Markdown, display

with RES.joinpath("resumen_metricas.csv").open(encoding="utf-8") as f:
    filas = list(csv.DictReader(f))
for r in filas:
    for c in ("p", "reps"):
        r[c] = int(r[c])
    for c in ("tiempo_medio_s", "tiempo_std_s", "speedup", "speedup_std", "eficiencia"):
        r[c] = float(r[c])

NOMBRES = {"numba": "Mem. compartida (numba)", "mpi": "Mem. distribuida (MPI)"}
lineas = ["| Modelo | p | Tiempo (s) | Speedup | Eficiencia |", "|---|---|---|---|---|"]
for r in filas:
    lineas.append(f"| {NOMBRES.get(r['modelo'], r['modelo'])} | {r['p']} | "
                  f"{r['tiempo_medio_s']:.3f} ± {r['tiempo_std_s']:.3f} | "
                  f"{r['speedup']:.2f} ± {r['speedup_std']:.2f} | {r['eficiencia']:.2f} |")
display(Markdown("\n".join(lineas)))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

for nombre in ("grafica_tiempo_vs_nucleos.png", "grafica_speedup_real_vs_ideal.png", "grafica_eficiencia.png"):
    fig, ax = plt.subplots(figsize=(9, 5.6))
    ax.imshow(mpimg.imread(RES / nombre))
    ax.axis("off")
    plt.show()

## 6. Cómo leer los resultados

**Tiempo vs. núcleos.** El tiempo baja de forma casi inversa a `p` mientras `p` no supera los núcleos físicos.
La línea punteada es `T1`, el secuencial.

**Speedup real vs. ideal.** El ideal es `S = p`. La distancia entre ambos es la sobrecarga: parte serial,
reducción/comunicación y límites de memoria.

**Eficiencia.** `E = S/p` cerca de 1 indica que casi no se pierde nada al sumar trabajadores.

**Por qué `p = 8` no se ajusta a Amdahl.** La laptop tiene **4 núcleos físicos con 8 hilos lógicos (SMT)**.
Amdahl supone procesadores idénticos e independientes; dos hilos SMT comparten unidades de cálculo y caché
L1/L2 de un mismo núcleo, así que el octavo hilo **no aporta un procesador completo más**. Por eso la fracción
paralela `f` se estima **solo con `p` ≤ núcleos físicos**, y `p = 8` se reporta aparte como caso de hilos
lógicos. Ajustar con `p = 8` incluido subestimaba `f` y dejaba la curva de Amdahl por debajo de los puntos reales.

In [ ]:
import numpy as np

fisicos = int(specs.get("nucleos_por_socket", 0)) * int(specs.get("sockets", 1) or 1) or (specs["nucleos_logicos"] // 2)

def ajustar_f(ps, ss):
    mejor_f, mejor_e = 0.0, float("inf")
    for f in np.linspace(0, 1, 10001):
        e = sum((1 / ((1 - f) + f / p) - s) ** 2 for p, s in zip(ps, ss))
        if e < mejor_e:
            mejor_f, mejor_e = float(f), e
    return mejor_f

print(f"Núcleos físicos: {fisicos}\n")
for modelo, nombre in NOMBRES.items():
    fm = sorted((r for r in filas if r["modelo"] == modelo), key=lambda r: r["p"])
    ajuste = [r for r in fm if 1 < r["p"] <= fisicos]
    if not ajuste:
        continue
    f = ajustar_f([r["p"] for r in ajuste], [r["speedup"] for r in ajuste])
    tope = float("inf") if f >= 1 else 1 / (1 - f)
    print(f"{nombre}: f = {f:.4f}  (ajuste con p = {[r['p'] for r in ajuste]})  ->  límite teórico p→∞ = {tope:.1f}")
    for r in fm:
        if r["p"] > 1:
            pred = 1 / ((1 - f) + f / r["p"])
            marca = "  (hilos lógicos SMT: fuera del ajuste)" if r["p"] > fisicos else ""
            print(f"    p={r['p']}: Amdahl predice {pred:.2f} | medido {r['speedup']:.2f}{marca}")

## 7. Interpretación y límites del análisis

- **Hasta los núcleos físicos el escalado es casi lineal**; lo que se pierde es la fracción serial pequeña
  (actualización de centroides y reducción) y el costo de comunicación/reparto en MPI.
- **Pasar a hilos lógicos aporta poco** (`p = 8` queda muy por debajo del ideal y de lo que Amdahl predice con
  núcleos reales): comparten unidades de cálculo y caché, y el recorrido de `X` está limitado por ancho de banda
  de memoria (los datos son mucho mayores que la caché L3).
- **El límite teórico `1/(1−f)` es una extrapolación** hecha con pocos puntos; en esta laptop el límite práctico
  lo fija el hardware (~4× por los núcleos físicos), no el algoritmo.
- **Memoria compartida vs. distribuida**: la diferencia de tiempos entre ambas es pequeña y, con 5 repeticiones,
  está dentro del ruido de medición; **no se declara un ganador**.
- **Ruido de medición**: es una laptop con turbo y límites térmicos, por eso se repite cada configuración y se
  reporta la desviación estándar. El secuencial, que fija `T1`, es muy estable.
- **Alcance**: el problema usa datos sintéticos y 25 iteraciones fijas; con datos reales o criterio de paro por
  convergencia el reparto entre fase paralela y serial podría cambiar.

## 8. Cómo reproducir

```bash
sudo apt install mpich libmpich-dev build-essential
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
python experimentos/verificar_correctitud.py
python experimentos/ejecutar_experimentos.py     # genera resultados/
```

También hay un `Dockerfile` con el entorno completo. Para abrir este notebook se necesita `ipykernel`
(`pip install ipykernel`) o la extensión de Jupyter de VS Code.